# Whisper — Robust Speech Recognition via Large-Scale Weak Supervision (2022)

_Papers · Speech & Audio_

**Take a plain encoder-decoder Transformer, train it on 680,000 hours of weakly-labelled internet audio, and prompt it with special tokens so one model transcribes, translates, and detects the language — zero-shot.**

---

This notebook is a practice scaffold for the **Whisper — Robust Speech Recognition via Large-Scale Weak Supervision (2022)** lesson. We build it up one small step at a time: first the multitask token format, then how flipping a single control token reassigns the task, then a conceptual look at the robustness result. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Define Whisper's four control-token slots

This is a **read-only conceptual demo**: we do not train or reproduce Whisper (its 680,000-hour weakly-supervised run is not reproducible here). Instead we recreate the special-token *interface* the decoder uses (paper §2.3). Every target sequence begins with a fixed prefix of four control slots — start-of-transcript, language, task, and timestamps — and those tokens are what select the job on one shared model.

In [ ]:
# The four control slots that prefix every Whisper target sequence (sec 2.3).
SOT = "<|startoftranscript|>"
EOT = "<|endoftext|>"

# The real model has 99 languages; we show three.
LANG_TOKENS = {"en": "<|en|>", "de": "<|de|>", "es": "<|es|>"}

# Task slot: same-language transcription vs. English translation.
TASK_TOKENS = {"transcribe": "<|transcribe|>", "translate": "<|translate|>"}

# Timestamp slot: plain text, else 20ms-quantized timestamp tokens.
TS_TOKENS = {"notimestamps": "<|notimestamps|>"}

### Step 2 — Assemble and label a target sequence

`build_sequence` lays the tokens out in the paper's exact order: the four-token control prefix, then the text tokens, then end-of-text. `parse_sequence` walks back over a sequence and names each control token's role, separating the prefix from the actual transcript. Reading these two functions makes the format concrete.

In [ ]:
def build_sequence(language, task, timestamps, text_tokens):
    """Assemble the decoder target sequence in the paper's order (sec 2.3)."""
    seq = [SOT, LANG_TOKENS[language], TASK_TOKENS[task]]
    # Slot 4: either real timestamp tokens or the no-timestamps marker.
    if timestamps:
        seq.append("<timestamps...>")
    else:
        seq.append(TS_TOKENS["notimestamps"])
    seq = seq + list(text_tokens) + [EOT]
    return seq

def parse_sequence(seq):
    """Label each control token's role; everything after the 4-token prefix is text."""
    roles = [
        "<|startoftranscript|> : start of transcript",
        f"{seq[1]:<22}: LANGUAGE  (predict this slot -> language identification / LID)",
        f"{seq[2]:<22}: TASK      (transcribe=same language | translate=English)",
        f"{seq[3]:<22}: TIMESTAMPS(notimestamps=plain text | else 20ms timestamp tokens)",
    ]
    text = [t for t in seq[4:] if t != EOT]
    return roles, text

### Step 3 — Decode one worked example

We build the sequence for a German speaker, transcribed, with no timestamps, producing the text "Guten Tag". Printing the decoded sequence and its parsed roles shows the four control tokens steering the output, followed by the two text tokens.

In [ ]:
# Worked example: German speaker, transcribe, no timestamps -> "Guten Tag".
seq = build_sequence("de", "transcribe", timestamps=False, text_tokens=["Guten", "Tag"])
print("decoded sequence:", seq)

roles, text = parse_sequence(seq)
print("\ncontrol-token roles:")
for r in roles:
    print("  " + r)

print("transcript text   :", " ".join(text))
print("number of control (prefix) tokens:", 4, " | text tokens:", len(text))
# decoded sequence: ['<|startoftranscript|>', '<|de|>', '<|transcribe|>', '<|notimestamps|>', 'Guten', 'Tag', '<|endoftext|>']
# transcript text   : Guten Tag

### Step 4 — Same model, different jobs

The point of the token interface: nothing about the weights changes between tasks — only the control prefix does. Here we flip the task token to `<|translate|>` to get an English translation of the same audio, and we leave the language slot unspecified so the model *predicts* it, which is exactly Whisper's zero-shot language identification.

In [ ]:
# Flip the TASK token: translate the SAME audio to English.
print("-- flip the TASK token: translate the SAME audio to English --")
print(build_sequence("es", "translate", timestamps=False, text_tokens=["Good", "morning"]))

# Leave the LANGUAGE slot for the model to PREDICT -> that prediction IS language ID.
print("-- leave the LANGUAGE slot for the model to PREDICT -> that prediction IS language ID --")
print("  prefix: ['<|startoftranscript|>', <model predicts <|xx|>>, '<|transcribe|>', '<|notimestamps|>', ...]")

# (Optional) actually transcribe with the released model. Commented: not preinstalled.
# !pip install -U openai-whisper
# import whisper
# model  = whisper.load_model("tiny")              # small Whisper checkpoint
# result = model.transcribe("audio.mp3")           # auto language ID + transcription
# print(result["text"])
# # To force translation to English instead: model.transcribe("audio.mp3", task="translate")

## Visualize the data & results

_Conceptual: how does Whisper's robustness (out-of-distribution error) compare to a supervised model matched to it in-distribution? (labeled illustration of the paper's sec 3.3 framing)_

### Step 1 — Set up the two models' illustrative error rates

These are **hand-chosen illustrative values**, not measured numbers from the paper — they only show the *shape* of the §3.3 result. We define two models that are matched in-distribution (same LibriSpeech word error rate) but differ out-of-distribution: the supervised model is tuned to LibriSpeech and degrades elsewhere, while Whisper stays robust.

In [ ]:
# CONCEPTUAL illustration of the paper's sec 3.3 robustness framing.
# These are HAND-CHOSEN illustrative values to show the SHAPE of the result,
# NOT measured numbers from the paper.
settings = ["in-distribution (LibriSpeech)", "out-of-distribution (avg)"]

# Supervised: great in-dist, degrades out-of-dist (illustrative).
supervised_wer = [3.0, 12.0]

# Whisper: zero-shot weak supervision, stays robust out-of-dist (illustrative).
whisper_wer = [3.0, 5.5]

### Step 2 — Report the robustness gap

With both models matched in-distribution, the story is the *out-of-distribution* gap: the supervised model's error roughly quadruples while Whisper's barely moves. That gap — not the shared LibriSpeech number — is the paper's robustness contribution.

In [ ]:
# Print each model's word error rate (WER) in both settings.
for name, wer in [("supervised", supervised_wer), ("whisper", whisper_wer)]:
    print(name, dict(zip(settings, wer)))

print("Same in-distribution WER; the out-of-distribution GAP is the robustness story (sec 3.3).")
print("NOTE: illustrative values only, not the paper's reported numbers.")

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The interface. You have one frozen Whisper model. A colleague wants three things from the same
            Spanish audio clip: (a) a Spanish transcript with no timing, (b) an English translation, and (c) for
            the model to tell them what language it is. Without retraining anything, how do you get all
            three, and what changes between them?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For (a): set the control prefix to <|startoftranscript|>, <|es|>, <|transcribe|>, <|notimestamps|>, then decode text. — _Forcing language=es and task=transcribe yields same-language (Spanish) output; the no-timestamps token suppresses timing._
- For (b): keep the same audio but set the task token to <|translate|>. — _<|translate|> outputs the English translation of the audio — only the task slot changes._
- For (c): leave the language token unspecified and read the highest-probability language token the decoder emits at that position. — _Predicting the language slot instead of forcing it IS Whisper's zero-shot language identification._

**Answer:** All three come from the same model by changing only the control tokens in the
                 decoder prefix. (a) Force <|es|> + <|transcribe|> +
                 <|notimestamps|> &rarr; plain Spanish transcript. (b) Switch the task token to
                 <|translate|> &rarr; English translation; nothing else changes. (c) Leave the
                 language token unforced and read the model's predicted language token &rarr; zero-shot
                 language identification. The weights never change — the token interface (§2.3) is the only
                 lever.

</details>

**Problem 2.** Robustness ablation (conceptual). A teammate argues: "Whisper just has a low LibriSpeech word
            error rate, so it's a good ASR model — nothing special." Using the paper's §3.3 framing, what
            comparison shows their reasoning is incomplete, and what would the result look like?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pick a supervised model whose LibriSpeech WER matches Whisper's — equal in-distribution accuracy. — _Controlling for in-distribution skill isolates robustness as the remaining variable._
- Evaluate both, zero-shot, on many out-of-distribution datasets (other accents, domains, noise) and average the WER. — _Robustness is about data the model was NOT tuned on; one benchmark cannot reveal it._
- Compare the averaged out-of-distribution WER of the two models. — _§3.3 reports Whisper makes far fewer errors here despite equal LibriSpeech WER — the gap IS the robustness contribution._

**Answer:** Equal LibriSpeech WER does not imply equal robustness. The paper's §3.3 comparison holds
                 in-distribution accuracy fixed — pick a supervised model that matches Whisper on
                 LibriSpeech — then measures average WER across many out-of-distribution datasets. There,
                 Whisper makes far fewer errors: same benchmark skill, much better generalization. That gap,
                 not the LibriSpeech number, is the contribution, and it comes from training on 680,000 hours of
                 varied weak supervision. Our CODEVIZ panel illustrates this gap qualitatively (our bars, not the
                 paper's numbers).

</details>

**Problem 3.** Format reasoning. Why did the authors put the task descriptor inside the decoder's token
            sequence (as <|transcribe|>/<|translate|> tokens) instead of, say,
            training a separate model per task or passing a task flag outside the sequence? Give the practical
            payoff.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that with tokens in the sequence, every example — any language, any task — is just "predict the next token" in one format. — _A single objective and one model can absorb all 680,000 hours uniformly; no per-task heads or separate training runs._
- Note the decoder conditions all later (text) tokens on the control prefix automatically. — _Self-attention over earlier tokens means the task descriptor steers generation with no architectural plumbing._
- Note you can force OR predict each control token at inference. — _Forcing gives controllability (set the task); predicting gives free abilities like language identification._

**Answer:** Encoding the task as ordinary tokens in the same sequence means one model and one
                 next-token objective cover transcription, translation, LID, and timing — so all 680,000 hours train
                 the same network uniformly, no per-task models or external flags. The decoder's causal
                 attention makes the control prefix automatically condition every text token, and at inference you
                 can force a control token to steer (set the task) or let the model predict it for
                 free zero-shot abilities (identify the language). That uniform, promptable interface is what let
                 weak supervision scale into one robust multitask model.

</details>